<a href="https://colab.research.google.com/github/Maee127/Adversarial-Notebooks/blob/master/%236/Notebook_6_Revised.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================
# Adversarial Attacks Series -- Note 06 (REVISED)
# Architectural Awareness: Building the Sensor into the Boat
# =============================================================
#
# Series:  Humble Model / Architectural Awareness
# Dataset: CIFAR-10
# Model:   CNN (same backbone as Essay #5)
#
#   Notebook structure:
#   Part A: Imports and Setup
#   Part B: Dataset Loading (CIFAR-10)
#   Part C: Model Architecture (shared backbone)
#   Part D: Train or load all three models
#   Part E: Attacks
#   Part F: Signal collection
#   Part G: Calibrate thresholds ON CALIB DATA ONLY
#   Part H: Evaluate each signal as a standalone gate
#   Part I: Fused gate
#   Part J: Summary table
# =============================================================
#
# CHANGES IN THIS REVISION (see chat for the full explanation of each):
#   1. RISK FORMULA FIXED. The previous version computed:
#        risk = 100 * ((total - correct) + deferred) / (total + deferred)
#      which counts every deferred sample as a failure -- a PERFECT
#      gate that defers exactly its own errors would still score
#      risk == deferral_rate under that formula, not ~0%. That
#      inverts this series' entire premise that deferral is the SAFE
#      choice. Risk is now `100 - accuracy(on predicted)`, matching
#      Essays #4 and #5 exactly, so numbers are comparable across the
#      series. Deferral rate and coverage are reported alongside it
#      as separate numbers, the way #4/#5 did -- not folded into risk.
#   2. clamp_valid() used in EVERY attack function. The previous
#      version used torch.clamp(x, 0, 1) in most of them, which is
#      wrong for CIFAR-normalized inputs -- exactly the bug Essay #5
#      fixed, that didn't carry over to this notebook.
#   3. The calibration split is now actually used. Every threshold
#      (confidence, disagreement, evidence) is calculated on
#      calib_loader and applied to eval_loader -- not calculated
#      in-sample on the same data being evaluated.
#   4. Multi-head's threshold is now calibrated (75th percentile of
#      disagreement on the calibration set), not a hardcoded 0.05
#      that never fired.
#   5. Boundary distance is computed for every image actually used in
#      the fused-gate sample -- no more padding unsampled images in a
#      batch with a copy of an unrelated image's score. Sample size
#      is controlled by evaluating fewer total images, not by
#      silently faking values within a batch.
#   6. Part H now includes a STANDALONE boundary-distance row, so its
#      individual contribution can be compared to confidence,
#      disagreement, and evidence before looking at the fused blend.
#   7. Fused gate uses Essay #5's OR-based fusion (defer if ANY
#      individual signal crosses ITS OWN calibrated threshold) instead
#      of a weighted linear blend of differently-scaled raw values.
#      The previous weighted-sum design mixed confidence (0-1, higher
#      = safer), disagreement (unbounded, higher = riskier), evidence
#      (unbounded, higher = safer), and boundary distance (0-1ish,
#      LOWER = riskier) into one number without a principled way to
#      make their scales comparable. OR-based fusion sidesteps that:
#      each signal only needs to be individually well-calibrated.
# =============================================================


In [ ]:
# -------------------------------------------------------------
# Part A: Imports and Setup
# -------------------------------------------------------------

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import os
from tqdm import tqdm
import random

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)


Using device: cuda


In [ ]:
# -------------------------------------------------------------
# Part B: Dataset Loading (CIFAR-10) with held-out split
# -------------------------------------------------------------

cifar_transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
cifar_transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=cifar_transform_train)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=cifar_transform_test)

calib_size = int(0.2 * len(test_dataset))
eval_size = len(test_dataset) - calib_size
calib_dataset, eval_dataset = random_split(test_dataset, [calib_size, eval_size],
                                            generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
calib_loader = DataLoader(calib_dataset, batch_size=64, shuffle=False, num_workers=0)
eval_loader = DataLoader(eval_dataset, batch_size=64, shuffle=False, num_workers=0)

print(f"Training samples: {len(train_dataset):,}")
print(f"Calibration samples: {len(calib_dataset):,}  <- thresholds come from here now")
print(f"Evaluation samples: {len(eval_dataset):,}  <- final numbers come from here")

CIFAR_MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
CIFAR_STD = torch.tensor([0.2023, 0.1994, 0.2010]).view(1, 3, 1, 1)
CIFAR_MIN = ((0 - CIFAR_MEAN) / CIFAR_STD).to(DEVICE)
CIFAR_MAX = ((1 - CIFAR_MEAN) / CIFAR_STD).to(DEVICE)

def clamp_valid(x):
    return torch.max(torch.min(x, CIFAR_MAX), CIFAR_MIN)


100%|██████████| 170M/170M [19:41<00:00, 144kB/s]


Training samples: 50,000
Calibration samples: 2,000  <- thresholds come from here now
Evaluation samples: 8,000  <- final numbers come from here


In [ ]:
# -------------------------------------------------------------
# Part C: Model Architecture (Shared Backbone)
# -------------------------------------------------------------

class BackboneCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = F.relu(self.conv1(x)); x = F.max_pool2d(x, 2)
        x = F.relu(self.conv2(x)); x = F.max_pool2d(x, 2)
        x = F.relu(self.conv3(x)); x = F.max_pool2d(x, 2)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return x

class StandardCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.backbone = BackboneCNN()
        self.fc_out = nn.Linear(256, num_classes)

    def forward(self, x):
        return self.fc_out(self.backbone(x))

class MultiHeadCNN(nn.Module):
    def __init__(self, backbone, num_heads=5, num_classes=10):
        super().__init__()
        self.backbone = backbone
        self.heads = nn.ModuleList([nn.Linear(256, num_classes) for _ in range(num_heads)])

    def forward(self, x):
        features = self.backbone(x)
        return torch.stack([head(features) for head in self.heads], dim=1)

    def forward_with_disagreement(self, x):
        logits = self.forward(x)
        probs = F.softmax(logits, dim=2)
        return probs.mean(dim=1), probs.var(dim=1)

class EvidentialCNN(nn.Module):
    def __init__(self, backbone, num_classes=10):
        super().__init__()
        self.backbone = backbone
        self.fc_alpha = nn.Linear(256, num_classes)

    def forward(self, x):
        features = self.backbone(x)
        return F.softplus(self.fc_alpha(features)) + 1.0

    def forward_with_evidence(self, x):
        alpha = self.forward(x)
        S = alpha.sum(dim=1, keepdim=True)
        return alpha / S, S.squeeze(1)

def evidential_loss(alpha, labels_onehot):
    S = alpha.sum(dim=1, keepdim=True)
    return (labels_onehot * (torch.digamma(S) - torch.digamma(alpha))).sum(dim=1).mean()


In [ ]:
# -------------------------------------------------------------
# Part D: Train or load all three models -- training loop unchanged,
# only the checkpoint filenames matter. If you already have
# checkpoints from the previous run, they're still valid (training
# wasn't buggy, only evaluation/attacks were) -- just point at them.
# -------------------------------------------------------------

def train_standard(model, loader, epochs=20):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    model.train()
    for epoch in range(epochs):
        for images, labels in tqdm(loader, desc=f"Epoch {epoch}"):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
    return model

def train_multihead(model, loader, epochs=20, num_heads=5):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    model.train()
    for epoch in range(epochs):
        for images, labels in tqdm(loader, desc=f"Epoch {epoch}"):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            logits = model(images)
            loss = torch.stack([criterion(logits[:, h, :], labels) for h in range(num_heads)]).mean()
            loss.backward()
            optimizer.step()
    return model

def train_evidential(model, loader, epochs=20):
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    model.train()
    for epoch in range(epochs):
        for images, labels in tqdm(loader, desc=f"Epoch {epoch}"):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            alpha = model(images)
            loss = evidential_loss(alpha, F.one_hot(labels, num_classes=10).float())
            loss.backward()
            optimizer.step()
    return model

def load_or_train(model, path, train_fn, loader):
    if os.path.exists(path):
        ckpt = torch.load(path, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])
        print(f"Loaded {path}")
    else:
        print(f"Training -> {path}")
        model = train_fn(model, loader)
        torch.save({'model_state_dict': model.state_dict()}, path)
    return model

baseline_model = load_or_train(StandardCNN().to(DEVICE), 'checkpoint_baseline_cifar10.pth', train_standard, train_loader)
multi_head_model = load_or_train(MultiHeadCNN(BackboneCNN(), num_heads=5).to(DEVICE), 'checkpoint_multihead.pth', train_multihead, train_loader)
evidential_model = load_or_train(EvidentialCNN(BackboneCNN()).to(DEVICE), 'checkpoint_evidential.pth', train_evidential, train_loader)

for m in [baseline_model, multi_head_model, evidential_model]:
    m.eval()


Training -> checkpoint_baseline_cifar10.pth


Epoch 19: 100%|██████████| 782/782 [00:24<00:00, 31.63it/s]


Training -> checkpoint_multihead.pth


Epoch 19: 100%|██████████| 782/782 [00:25<00:00, 30.23it/s]


Training -> checkpoint_evidential.pth


Epoch 19: 100%|██████████| 782/782 [00:25<00:00, 31.16it/s]


In [ ]:
# -------------------------------------------------------------
# Part E: Attacks -- ONE set of correct attack functions, reused by
# everything below. clamp_valid() everywhere, no exceptions.
# -------------------------------------------------------------

def fgsm_standard(model, images, labels, epsilon=0.03):
    model.eval()
    images = images.clone().detach().requires_grad_(True)
    loss = nn.CrossEntropyLoss()(model(images), labels)
    model.zero_grad(); loss.backward()
    return clamp_valid(images + epsilon * images.grad.sign()).detach()

def pgd_standard(model, images, labels, epsilon=0.03, step_size=0.007, num_steps=40):
    model.eval()
    images_adv = images.clone().detach()
    for _ in range(num_steps):
        images_adv.requires_grad_(True)
        loss = nn.CrossEntropyLoss()(model(images_adv), labels)
        model.zero_grad(); loss.backward()
        with torch.no_grad():
            images_adv = images_adv + step_size * images_adv.grad.sign()
            perturbation = torch.clamp(images_adv - images, -epsilon, epsilon)
            images_adv = clamp_valid(images + perturbation)
    return images_adv.detach()

def _multihead_loss(model, images, labels):
    logits = model(images)
    criterion = nn.CrossEntropyLoss()
    return torch.stack([criterion(logits[:, h, :], labels) for h in range(logits.size(1))]).mean()

def fgsm_multihead(model, images, labels, epsilon=0.03):
    model.eval()
    images = images.clone().detach().requires_grad_(True)
    loss = _multihead_loss(model, images, labels)
    model.zero_grad(); loss.backward()
    return clamp_valid(images + epsilon * images.grad.sign()).detach()

def pgd_multihead(model, images, labels, epsilon=0.03, step_size=0.007, num_steps=40):
    model.eval()
    images_adv = images.clone().detach()
    for _ in range(num_steps):
        images_adv.requires_grad_(True)
        loss = _multihead_loss(model, images_adv, labels)
        model.zero_grad(); loss.backward()
        with torch.no_grad():
            images_adv = images_adv + step_size * images_adv.grad.sign()
            perturbation = torch.clamp(images_adv - images, -epsilon, epsilon)
            images_adv = clamp_valid(images + perturbation)
    return images_adv.detach()

def fgsm_evidential(model, images, labels, epsilon=0.03):
    model.eval()
    images = images.clone().detach().requires_grad_(True)
    alpha = model(images)
    loss = nn.CrossEntropyLoss()(alpha, labels)
    model.zero_grad(); loss.backward()
    return clamp_valid(images + epsilon * images.grad.sign()).detach()

def pgd_evidential(model, images, labels, epsilon=0.03, step_size=0.007, num_steps=40):
    model.eval()
    images_adv = images.clone().detach()
    for _ in range(num_steps):
        images_adv.requires_grad_(True)
        alpha = model(images_adv)
        loss = nn.CrossEntropyLoss()(alpha, labels)
        model.zero_grad(); loss.backward()
        with torch.no_grad():
            images_adv = images_adv + step_size * images_adv.grad.sign()
            perturbation = torch.clamp(images_adv - images, -epsilon, epsilon)
            images_adv = clamp_valid(images + perturbation)
    return images_adv.detach()

def estimate_boundary_distance(model, image, max_iters=30):
    model.eval()

    with torch.no_grad():
        start_pred = model(image).argmax(dim=1).item()

    x = image.clone().detach().requires_grad_(True)

    loss = nn.CrossEntropyLoss()(
        model(x),
        torch.tensor([start_pred], device=DEVICE),
    )

    model.zero_grad()
    loss.backward()

    direction = x.grad.detach().sign()

    low, high = 0.0, 1.0

    for _ in range(max_iters):
        mid = (low + high) / 2
        perturbed = clamp_valid(image + mid * direction)

        with torch.no_grad():
            pred = model(perturbed).argmax(dim=1).item()

        if pred != start_pred:
            high = mid
        else:
            low = mid

    return (low + high) / 2
PGD_PARAMS = {
    "epsilon": 0.03,
    "step_size": 0.007,
    "num_steps": 40,
}



In [ ]:
# -------------------------------------------------------------
# Part F: Signal collection
#
# Signals:
#   - baseline confidence
#   - multi-head disagreement
#   - evidential evidence
#   - boundary distance
#
# Every output array contains exactly one entry per image.
# Unsampled boundary values are represented by:
#   boundary       = np.nan
#   boundary_valid = False
# -------------------------------------------------------------

def collect_signals(
    baseline,
    multihead,
    evidential,
    loader,
    condition="clean",
    n_samples=None,
    n_boundary_samples=300,
    boundary_seed=42,
):
    """
    Collect all model signals while preserving per-image alignment.

    condition:
        "clean" or "adversarial"

    n_samples:
        Optional limit on the number of processed images.

    n_boundary_samples:
        Number of images receiving the expensive boundary-distance
        calculation.

    boundary_seed:
        Controls the random boundary-distance subset. Use the same seed
        for clean and adversarial conditions when comparing the same images.
    """

    if condition not in {"clean", "adversarial"}:
        raise ValueError(
            "condition must be either 'clean' or 'adversarial'"
        )

    total_available = (
        n_samples
        if n_samples is not None
        else len(loader.dataset)
    )

    rng = np.random.RandomState(boundary_seed)

    boundary_indices = set(
        rng.choice(
            total_available,
            size=min(n_boundary_samples, total_available),
            replace=False,
        ).tolist()
    )

    out = {
        "confidence": [],
        "disagreement": [],
        "evidence": [],
        "boundary": [],
        "boundary_valid": [],
        "sample_index": [],
        "correct_baseline": [],
        "correct_multihead": [],
        "correct_evidential": [],
    }

    n_seen = 0
    n_boundary_done = 0

    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        if condition == "adversarial":
            images = pgd_standard(
                baseline,
                images,
                labels,
                **PGD_PARAMS,
            )

        with torch.no_grad():
            # ---------------------------------------------------------
            # Baseline confidence
            # ---------------------------------------------------------
            base_logits = baseline(images)
            base_probs = F.softmax(base_logits, dim=1)
            base_conf, base_pred = base_probs.max(dim=1)

            # ---------------------------------------------------------
            # Multi-head disagreement
            # ---------------------------------------------------------
            mean_probs, var_probs = (
                multihead.forward_with_disagreement(images)
            )

            mh_pred = mean_probs.argmax(dim=1)
            disagreement = var_probs.max(dim=1)[0]

            # ---------------------------------------------------------
            # Evidential output
            # ---------------------------------------------------------
            ev_probs, evidence = (
                evidential.forward_with_evidence(images)
            )

            ev_pred = ev_probs.argmax(dim=1)

        for i in range(images.size(0)):
            if n_samples is not None and n_seen >= n_samples:
                break

            # One entry per image for every signal
            out["confidence"].append(
                base_conf[i].item()
            )

            out["disagreement"].append(
                disagreement[i].item()
            )

            out["evidence"].append(
                evidence[i].item()
            )

            out["correct_baseline"].append(
                int(base_pred[i].item() == labels[i].item())
            )

            out["correct_multihead"].append(
                int(mh_pred[i].item() == labels[i].item())
            )

            out["correct_evidential"].append(
                int(ev_pred[i].item() == labels[i].item())
            )

            out["sample_index"].append(n_seen)

            # ---------------------------------------------------------
            # Boundary distance
            # ---------------------------------------------------------
            if n_seen in boundary_indices:
                boundary_distance = (
                    estimate_boundary_distance(
                        baseline,
                        images[i:i + 1],
                        max_iters=30,
                    )
                )

                out["boundary"].append(
                    boundary_distance
                )

                out["boundary_valid"].append(True)

                n_boundary_done += 1

                if n_boundary_done % 100 == 0:
                    print(
                        f"   ...boundary distance: "
                        f"{n_boundary_done}/"
                        f"{len(boundary_indices)}"
                    )

            else:
                # Preserve alignment for unsampled images
                out["boundary"].append(np.nan)
                out["boundary_valid"].append(False)

            n_seen += 1

        if n_samples is not None and n_seen >= n_samples:
            break

    # -------------------------------------------------------------
    # Convert outputs to aligned NumPy arrays
    # -------------------------------------------------------------
    out["confidence"] = np.asarray(
        out["confidence"],
        dtype=float,
    )

    out["disagreement"] = np.asarray(
        out["disagreement"],
        dtype=float,
    )

    out["evidence"] = np.asarray(
        out["evidence"],
        dtype=float,
    )

    out["boundary"] = np.asarray(
        out["boundary"],
        dtype=float,
    )

    out["boundary_valid"] = np.asarray(
        out["boundary_valid"],
        dtype=bool,
    )

    out["sample_index"] = np.asarray(
        out["sample_index"],
        dtype=int,
    )

    out["correct_baseline"] = np.asarray(
        out["correct_baseline"],
        dtype=bool,
    )

    out["correct_multihead"] = np.asarray(
        out["correct_multihead"],
        dtype=bool,
    )

    out["correct_evidential"] = np.asarray(
        out["correct_evidential"],
        dtype=bool,
    )

    # -------------------------------------------------------------
    # Alignment validation
    # -------------------------------------------------------------
    expected_length = len(out["confidence"])

    for key, value in out.items():
        assert len(value) == expected_length, (
            f"{key} has length {len(value)}, "
            f"expected {expected_length}"
        )

    assert (
        out["boundary_valid"].sum()
        == n_boundary_done
    )

    print(
        f"Collected {expected_length} images "
        f"({n_boundary_done} boundary measurements)."
    )

    return out


# =============================================================
# Run signal collection
# =============================================================

print(
    "Collecting calibration signals "
    "(clean + adversarial)..."
)

CALIB_BOUNDARY_SEED = 44

calib_clean = collect_signals(
    baseline_model,
    multi_head_model,
    evidential_model,
    calib_loader,
    condition="clean",
    n_boundary_samples=400,
    boundary_seed=CALIB_BOUNDARY_SEED,
)

calib_adv = collect_signals(
    baseline_model,
    multi_head_model,
    evidential_model,
    calib_loader,
    condition="adversarial",
    n_boundary_samples=400,
    boundary_seed=CALIB_BOUNDARY_SEED,
)


print(
    "\nCollecting evaluation signals "
    "(clean + adversarial)..."
)

EVAL_BOUNDARY_SEED = 44

eval_clean = collect_signals(
    baseline_model,
    multi_head_model,
    evidential_model,
    eval_loader,
    condition="clean",
    n_boundary_samples=1200,
    boundary_seed=EVAL_BOUNDARY_SEED,
)

eval_adv = collect_signals(
    baseline_model,
    multi_head_model,
    evidential_model,
    eval_loader,
    condition="adversarial",
    n_boundary_samples=1200,
    boundary_seed=EVAL_BOUNDARY_SEED,
)


# =============================================================
# Final shape verification
# =============================================================

print("\nFinal shape verification:")

for name, data in [
    ("calib_clean", calib_clean),
    ("calib_adv", calib_adv),
    ("eval_clean", eval_clean),
    ("eval_adv", eval_adv),
]:
    print(
        f"{name}: "
        f"images={len(data['confidence'])}, "
        f"boundary_valid="
        f"{int(data['boundary_valid'].sum())}"
    )


   ...boundary distance: 100/400
   ...boundary distance: 200/400
   ...boundary distance: 300/400
   ...boundary distance: 400/400
Collected 2000 images (400 boundary measurements).
   ...boundary distance: 100/400
   ...boundary distance: 200/400
   ...boundary distance: 300/400
   ...boundary distance: 400/400
Collected 2000 images (400 boundary measurements).

   ...boundary distance: 100/1200
   ...boundary distance: 200/1200
   ...boundary distance: 300/1200
   ...boundary distance: 400/1200
   ...boundary distance: 500/1200
   ...boundary distance: 600/1200
   ...boundary distance: 700/1200
   ...boundary distance: 800/1200
   ...boundary distance: 900/1200
   ...boundary distance: 1000/1200
   ...boundary distance: 1100/1200
   ...boundary distance: 1200/1200
Collected 8000 images (1200 boundary measurements).
   ...boundary distance: 100/1200
   ...boundary distance: 200/1200
   ...boundary distance: 300/1200
   ...boundary distance: 400/1200
   ...boundary distance: 500/1200


In [ ]:
# -------------------------------------------------------------
# Part G: Calibrate thresholds ON CALIB DATA ONLY
# -------------------------------------------------------------

confidence_threshold = np.percentile(
    calib_clean["confidence"],
    25,
)

disagreement_threshold = np.percentile(
    calib_clean["disagreement"],
    75,
)

evidence_threshold = np.percentile(
    calib_clean["evidence"],
    25,
)

boundary_values = calib_clean["boundary"][
    calib_clean["boundary_valid"]
]

boundary_threshold = np.percentile(
    boundary_values,
    25,
)

confidence_fragility_threshold = 1 - confidence_threshold

print(
    f"\nConfidence threshold: "
    f"{confidence_threshold:.4f} "
    f"(low confidence = fragile)"
)

print(
    f"Confidence fragility threshold: "
    f"{confidence_fragility_threshold:.4f}"
)

print(
    f"   Confidence:   {confidence_threshold:.4f} "
    f"(25th percentile; low = fragile)"
)

print(
    f"   Disagreement: "
    f"{disagreement_threshold:.12e} "
    f"(75th percentile; high = fragile)"
)

print(
    f"   Evidence:     {evidence_threshold:.4f} "
    f"(25th percentile; low = fragile)"
)

print(
    f"   Boundary:     {boundary_threshold:.4f} "
    f"(25th percentile; low = fragile)"
)
disagreement_values = calib_clean["disagreement"]

print(
    "Disagreement statistics:"
)

print(
    f"min={disagreement_values.min():.8f}, "
    f"median={np.median(disagreement_values):.8f}, "
    f"mean={disagreement_values.mean():.8f}, "
    f"max={disagreement_values.max():.8f}"
)

print(
    "exact zeros:",
    np.sum(disagreement_values == 0),
    "/",
    len(disagreement_values),
)

print(
    "nonzero values:",
    np.sum(disagreement_values > 0),
    "/",
    len(disagreement_values),
)




Confidence threshold: 0.6341 (low confidence = fragile)
Confidence fragility threshold: 0.3659
   Confidence:   0.6341 (25th percentile; low = fragile)
   Disagreement: 2.810709673895e-07 (75th percentile; high = fragile)
   Evidence:     210.8431 (25th percentile; low = fragile)
   Boundary:     0.0086 (25th percentile; low = fragile)
Disagreement statistics:
min=0.00000000, median=0.00000004, mean=0.00000082, max=0.00007163
exact zeros: 0 / 2000
nonzero values: 2000 / 2000


In [ ]:
# -------------------------------------------------------------
# Part H: Evaluate each signal as a standalone gate, Essay-4/5-style
# (coverage / accuracy-on-predicted / deferral / risk), including a
# STANDALONE boundary-distance row that was missing before.
# -------------------------------------------------------------

def gate_metrics(
    scores,
    correct,
    threshold,
    higher_is_fragile=True,
    valid_mask=None,
):
    scores = np.asarray(scores, dtype=float)
    correct = np.asarray(correct, dtype=bool)

    if valid_mask is not None:
        valid_mask = np.asarray(valid_mask, dtype=bool)
        scores = scores[valid_mask]
        correct = correct[valid_mask]

    defer_mask = (
        scores > threshold
        if higher_is_fragile
        else scores < threshold
    )

    predict_mask = ~defer_mask

    n_total = len(scores)
    n_predicted = predict_mask.sum()
    n_deferred = defer_mask.sum()

    accuracy = (
        100 * correct[predict_mask].sum() / n_predicted
        if n_predicted > 0
        else float("nan")
    )

    return {
        "coverage": 100 * n_predicted / n_total,
        "accuracy": accuracy,
        "deferral_rate": 100 * n_deferred / n_total,
        "risk": 100 - accuracy,
    }

def signal_report(
    name,
    clean_scores,
    clean_correct,
    adv_scores,
    adv_correct,
    threshold,
    higher_is_fragile=True,
    clean_valid=None,
    adv_valid=None,
):
    c = gate_metrics(
        clean_scores,
        clean_correct,
        threshold,
        higher_is_fragile,
        clean_valid,
    )

    a = gate_metrics(
        adv_scores,
        adv_correct,
        threshold,
        higher_is_fragile,
        adv_valid,
    )

    print(f"\n{name} (threshold={threshold:.12e}):")
    print(
        f"   Clean: coverage={c['coverage']:.2f}% "
        f"accuracy={c['accuracy']:.2f}% "
        f"deferral={c['deferral_rate']:.2f}% "
        f"risk={c['risk']:.2f}%"
    )
    print(
        f"   Adversarial: coverage={a['coverage']:.2f}% "
        f"accuracy={a['accuracy']:.2f}% "
        f"deferral={a['deferral_rate']:.2f}% "
        f"risk={a['risk']:.2f}%"
    )

    return c, a


print("\n" + "=" * 60)
print("STANDALONE SIGNAL COMPARISON (calibrated on held-out split)")
print("=" * 60)

conf_clean, conf_adv = signal_report(
    "Confidence (baseline)",
    1 - eval_clean['confidence'],
    eval_clean['correct_baseline'],
    1 - eval_adv['confidence'],
    eval_adv['correct_baseline'],
    1 - confidence_threshold, True,
    )

disagree_clean, disagree_adv = signal_report(
    "Disagreement (multi-head)",
    eval_clean['disagreement'],
    eval_clean['correct_multihead'],
    eval_adv['disagreement'],
    eval_adv['correct_multihead'],
    disagreement_threshold,
    True,
    )

evidence_clean, evidence_adv = signal_report(
    "Evidence (evidential)",
    eval_clean['evidence'],
    eval_clean['correct_evidential'],
    eval_adv['evidence'],
    eval_adv['correct_evidential'],
    evidence_threshold,
    False,
    )  # LOW evidence = fragile

boundary_clean, boundary_adv = signal_report(
    "Boundary distance",
    eval_clean["boundary"],
    eval_clean["correct_baseline"],
    eval_adv["boundary"],
    eval_adv["correct_baseline"],
    boundary_threshold,
    False,
    eval_clean["boundary_valid"],
    eval_adv["boundary_valid"],
)



STANDALONE SIGNAL COMPARISON (calibrated on held-out split)

Confidence (baseline) (threshold=3.658820688725e-01):
   Clean: coverage=75.47% accuracy=89.96% deferral=24.52% risk=10.04%
   Adversarial: coverage=76.89% accuracy=22.29% deferral=23.11% risk=77.71%

Disagreement (multi-head) (threshold=2.810709673895e-07):
   Clean: coverage=74.92% accuracy=86.30% deferral=25.07% risk=13.70%
   Adversarial: coverage=71.83% accuracy=81.08% deferral=28.18% risk=18.92%

Evidence (evidential) (threshold=2.108430786133e+02):
   Clean: coverage=75.56% accuracy=84.53% deferral=24.44% risk=15.47%
   Adversarial: coverage=73.94% accuracy=79.36% deferral=26.06% risk=20.64%

Boundary distance (threshold=8.595540653914e-03):
   Clean: coverage=72.33% accuracy=88.02% deferral=27.67% risk=11.98%
   Adversarial: coverage=71.50% accuracy=21.68% deferral=28.50% risk=78.32%


In [ ]:
# -------------------------------------------------------------
# Part I: Fused gate -- OR-based (Essay #5 style), not a weighted
# blend of differently-scaled raw signals.
# -------------------------------------------------------------

def fused_or_gate_metrics(conf, disagreement, evidence, boundary, correct,
                           conf_thresh, dis_thresh, ev_thresh, bd_thresh, use_boundary=True):
    n = len(conf)
    boundary = np.array([b if b is not None else np.nan for b in boundary], dtype=float)
    defer = (conf < conf_thresh) | (disagreement > dis_thresh) | (evidence < ev_thresh)
    if use_boundary:
        has_boundary = ~np.isnan(boundary)
        defer = defer | (has_boundary & (boundary < bd_thresh))
    predict_mask = ~defer
    n_predicted = predict_mask.sum()
    acc = 100 * correct[predict_mask].sum() / n_predicted if n_predicted > 0 else float('nan')
    return {
        'coverage': 100 * n_predicted / n,
        'accuracy': acc,
        'deferral_rate': 100 * defer.sum() / n,
        'risk': 100 - acc if not np.isnan(acc) else float('nan'),
    }

# Use baseline's own correctness as ground truth for the fused gate,
# since it's the model actually making the final prediction.
fused_clean = fused_or_gate_metrics(
    eval_clean['confidence'], eval_clean['disagreement'], eval_clean['evidence'],
    eval_clean['boundary'], eval_clean['correct_baseline'],
    confidence_threshold, disagreement_threshold, evidence_threshold, boundary_threshold
)
fused_adv = fused_or_gate_metrics(
    eval_adv['confidence'], eval_adv['disagreement'], eval_adv['evidence'],
    eval_adv['boundary'], eval_adv['correct_baseline'],
    confidence_threshold, disagreement_threshold, evidence_threshold, boundary_threshold
)

print("\n" + "=" * 60)
print("FUSED GATE (OR-based: confidence OR disagreement OR evidence OR boundary)")
print("=" * 60)
print(f"Clean:       {fused_clean}")
print(f"Adversarial: {fused_adv}")

# For comparison: fused WITHOUT boundary distance, to isolate its contribution
fused_clean_nobd = fused_or_gate_metrics(
    eval_clean['confidence'], eval_clean['disagreement'], eval_clean['evidence'],
    eval_clean['boundary'], eval_clean['correct_baseline'],
    confidence_threshold, disagreement_threshold, evidence_threshold, boundary_threshold, use_boundary=False
)
fused_adv_nobd = fused_or_gate_metrics(
    eval_adv['confidence'], eval_adv['disagreement'], eval_adv['evidence'],
    eval_adv['boundary'], eval_adv['correct_baseline'],
    confidence_threshold, disagreement_threshold, evidence_threshold, boundary_threshold, use_boundary=False
)
print("\nFused gate WITHOUT boundary distance (confidence OR disagreement OR evidence only):")
print(f"Clean:       {fused_clean_nobd}")
print(f"Adversarial: {fused_adv_nobd}")
print("\nCompare the two fused rows above -- the gap between them is boundary")
print("distance's actual marginal contribution, which the previous weighted-sum")
print("design couldn't isolate.")



FUSED GATE (OR-based: confidence OR disagreement OR evidence OR boundary)
Clean:       {'coverage': np.float64(54.6125), 'accuracy': np.float64(94.87296864271), 'deferral_rate': np.float64(45.3875), 'risk': np.float64(5.127031357289994)}
Adversarial: {'coverage': np.float64(43.825), 'accuracy': np.float64(37.10781517398745), 'deferral_rate': np.float64(56.175), 'risk': np.float64(62.89218482601255)}

Fused gate WITHOUT boundary distance (confidence OR disagreement OR evidence only):
Clean:       {'coverage': np.float64(55.1375), 'accuracy': np.float64(94.6270686919066), 'deferral_rate': np.float64(44.8625), 'risk': np.float64(5.372931308093399)}
Adversarial: {'coverage': np.float64(44.8), 'accuracy': np.float64(36.746651785714285), 'deferral_rate': np.float64(55.2), 'risk': np.float64(63.253348214285715)}

Compare the two fused rows above -- the gap between them is boundary
distance's actual marginal contribution, which the previous weighted-sum
design couldn't isolate.


In [ ]:
# -------------------------------------------------------------
# Part J: Summary table -- all numbers use the SAME risk definition
# -------------------------------------------------------------

print("\n" + "=" * 70)
print(f"{'Method':<32}{'Clean cov':>10}{'Clean acc':>11}{'Clean risk':>12}"
      f"{'Adv cov':>10}{'Adv acc':>10}{'Adv risk':>10}")
print("-" * 70)
rows = [
    ("Confidence (baseline)", conf_clean, conf_adv),
    ("Disagreement (multi-head)", disagree_clean, disagree_adv),
    ("Evidence (evidential)", evidence_clean, evidence_adv),
    ("Boundary distance (standalone)", boundary_clean, boundary_adv),
    ("Fused (no boundary)", fused_clean_nobd, fused_adv_nobd),
    ("Fused (with boundary)", fused_clean, fused_adv),
]
for name, c, a in rows:
    print(f"{name:<32}{c['coverage']:>9.2f}%{c['accuracy']:>10.2f}%{c['risk']:>11.2f}%"
          f"{a['coverage']:>9.2f}%{a['accuracy']:>9.2f}%{a['risk']:>9.2f}%")

print("\n" + "!" * 70)
print(
    "NOTE: The standalone boundary-distance row is evaluated "
    "on the 1,200 images with valid boundary measurements, "
    "while the other standalone rows use all 8,000 evaluation images."
)
print(
    "Therefore, Part K is the primary apples-to-apples comparison."
)
print("!" * 70)




Method                           Clean cov  Clean acc  Clean risk   Adv cov   Adv acc  Adv risk
----------------------------------------------------------------------
Confidence (baseline)               75.47%     89.96%      10.04%    76.89%    22.29%    77.71%
Disagreement (multi-head)           74.92%     86.30%      13.70%    71.83%    81.08%    18.92%
Evidence (evidential)               75.56%     84.53%      15.47%    73.94%    79.36%    20.64%
Boundary distance (standalone)      72.33%     88.02%      11.98%    71.50%    21.68%    78.32%
Fused (no boundary)                 55.14%     94.63%       5.37%    44.80%    36.75%    63.25%
Fused (with boundary)               54.61%     94.87%       5.13%    43.83%    37.11%    62.89%

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
NOTE: The standalone boundary-distance row is evaluated on the 1,200 images with valid boundary measurements, while the other standalone rows use all 8,000 evaluation images.
Therefore

In [ ]:
# -------------------------------------------------------------
# Part K: FAIR comparison
# All signals are evaluated on exactly the same images for which
# boundary distance was successfully computed.
# -------------------------------------------------------------

def subset_data(data, mask):
    mask = np.asarray(mask, dtype=bool)
    n = len(mask)

    return {
        key: np.asarray(value)[mask]
        for key, value in data.items()
        if len(value) == n
    }


clean_boundary_valid = np.asarray(
    eval_clean["boundary_valid"],
    dtype=bool,
)

adv_boundary_valid = np.asarray(
    eval_adv["boundary_valid"],
    dtype=bool,
)

# Same underlying evaluation images must have valid boundary scores
# in both clean and adversarial conditions.
shared_boundary_mask = (
    clean_boundary_valid
    & adv_boundary_valid
)

N_SHARED_BOUNDARY = int(shared_boundary_mask.sum())

print(
    f"\nRestricting fair comparison to "
    f"{N_SHARED_BOUNDARY} identical images "
    f"with valid boundary-distance scores."
)

fair_clean = subset_data(
    eval_clean,
    shared_boundary_mask,
)

fair_adv = subset_data(
    eval_adv,
    shared_boundary_mask,
)

print("\n" + "=" * 60)
print("FAIR STANDALONE SIGNAL COMPARISON")
print("=" * 60)

fair_conf_clean, fair_conf_adv = signal_report(
    "Confidence (baseline)",
    1 - fair_clean["confidence"],
    fair_clean["correct_baseline"],
    1 - fair_adv["confidence"],
    fair_adv["correct_baseline"],
    1 - confidence_threshold,
    True,
)

fair_disagree_clean, fair_disagree_adv = signal_report(
    "Disagreement (multi-head)",
    fair_clean["disagreement"],
    fair_clean["correct_multihead"],
    fair_adv["disagreement"],
    fair_adv["correct_multihead"],
    disagreement_threshold,
    True,
)

fair_evidence_clean, fair_evidence_adv = signal_report(
    "Evidence (evidential)",
    fair_clean["evidence"],
    fair_clean["correct_evidential"],
    fair_adv["evidence"],
    fair_adv["correct_evidential"],
    evidence_threshold,
    False,
)

fair_boundary_clean, fair_boundary_adv = signal_report(
    "Boundary distance",
    fair_clean["boundary"],
    fair_clean["correct_baseline"],
    fair_adv["boundary"],
    fair_adv["correct_baseline"],
    boundary_threshold,
    False,
)

print("\n" + "=" * 80)
print(
    f"{'Method (same sample, n=' + str(N_SHARED_BOUNDARY) + ')':<38}"
    f"{'Clean cov':>10}"
    f"{'Clean acc':>11}"
    f"{'Clean risk':>12}"
    f"{'Adv cov':>10}"
    f"{'Adv acc':>10}"
    f"{'Adv risk':>10}"
)
print("-" * 80)

fair_rows = [
    ("Confidence (baseline)", fair_conf_clean, fair_conf_adv),
    ("Disagreement (multi-head)", fair_disagree_clean, fair_disagree_adv),
    ("Evidence (evidential)", fair_evidence_clean, fair_evidence_adv),
    ("Boundary distance", fair_boundary_clean, fair_boundary_adv),
]

for name, clean_metrics, adv_metrics in fair_rows:
    print(
        f"{name:<38}"
        f"{clean_metrics['coverage']:>9.2f}%"
        f"{clean_metrics['accuracy']:>10.2f}%"
        f"{clean_metrics['risk']:>11.2f}%"
        f"{adv_metrics['coverage']:>9.2f}%"
        f"{adv_metrics['accuracy']:>9.2f}%"
        f"{adv_metrics['risk']:>9.2f}%"
    )

print(
    "\nThis table is the valid apples-to-apples comparison: "
    "every signal is judged on the same images."
)


Restricting fair comparison to 1200 identical images with valid boundary-distance scores.

FAIR STANDALONE SIGNAL COMPARISON

Confidence (baseline) (threshold=3.658820688725e-01):
   Clean: coverage=73.75% accuracy=89.15% deferral=26.25% risk=10.85%
   Adversarial: coverage=75.75% accuracy=19.91% deferral=24.25% risk=80.09%

Disagreement (multi-head) (threshold=2.810709673895e-07):
   Clean: coverage=73.92% accuracy=85.23% deferral=26.08% risk=14.77%
   Adversarial: coverage=71.75% accuracy=78.98% deferral=28.25% risk=21.02%

Evidence (evidential) (threshold=2.108430786133e+02):
   Clean: coverage=73.67% accuracy=83.71% deferral=26.33% risk=16.29%
   Adversarial: coverage=72.75% accuracy=78.12% deferral=27.25% risk=21.88%

Boundary distance (threshold=8.595540653914e-03):
   Clean: coverage=72.33% accuracy=88.02% deferral=27.67% risk=11.98%
   Adversarial: coverage=71.50% accuracy=21.68% deferral=28.50% risk=78.32%

Method (same sample, n=1200)           Clean cov  Clean acc  Clean ri

In [ ]:
# -------------------------------------------------------------
# Part L: Coverage-matched comparison.
#
# Part K evaluates each sensor at its own calibrated operating point.
# Their coverage values are close but not identical. This section
# adjusts the confidence, disagreement, and evidence thresholds on the
# separate adversarial calibration set so their evaluation coverage
# approximately matches boundary distance.
#
# This is a secondary diagnostic. Part K remains the primary fair
# comparison because its thresholds come from clean calibration data.
# -------------------------------------------------------------

def find_threshold_for_coverage(calib_scores, target_coverage_pct, higher_is_fragile=True):
    """Binary-search a threshold on calibration scores so that
    deferral rate matches (100 - target_coverage_pct)."""
    target_deferral_pct = 100 - target_coverage_pct
    lo, hi = float(np.min(calib_scores)), float(np.max(calib_scores))
    for _ in range(50):
        mid = (lo + hi) / 2
        defer_pct = 100 * (calib_scores > mid).mean() if higher_is_fragile else 100 * (calib_scores < mid).mean()
        if defer_pct > target_deferral_pct:
            if higher_is_fragile:
                lo = mid
            else:
                hi = mid
        else:
            if higher_is_fragile:
                hi = mid
            else:
                lo = mid
    return (lo + hi) / 2

target_coverage = fair_boundary_adv['coverage']
print(f"\nTarget adversarial coverage (boundary distance's own operating point): {target_coverage:.2f}%")

conf_matched_thresh = find_threshold_for_coverage(1 - calib_adv['confidence'], target_coverage, True)
disagree_matched_thresh = find_threshold_for_coverage(calib_adv['disagreement'], target_coverage, True)
evidence_matched_thresh = find_threshold_for_coverage(calib_adv['evidence'], target_coverage, False)

print(f"Coverage-matched thresholds (calibrated on calib_adv):")
print(f"   Confidence:   {conf_matched_thresh:.4f}  (vs. own threshold {1 - confidence_threshold:.4f})")
print(f"   Disagreement: {disagree_matched_thresh:.12f}  (vs. own threshold {disagreement_threshold:.12f})")
print(f"   Evidence:     {evidence_matched_thresh:.4f}  (vs. own threshold {evidence_threshold:.4f})")

matched_conf_adv = gate_metrics(1 - fair_adv['confidence'], fair_adv['correct_baseline'], conf_matched_thresh, True)
matched_disagree_adv = gate_metrics(fair_adv['disagreement'], fair_adv['correct_multihead'], disagree_matched_thresh, True)
matched_evidence_adv = gate_metrics(fair_adv['evidence'], fair_adv['correct_evidential'], evidence_matched_thresh, False)

print("\n" + "=" * 70)
print(f"{'Method (adversarial, coverage-matched to ~' + f'{target_coverage:.0f}%)':<45}"
      f"{'Coverage':>10}{'Accuracy':>10}{'Risk':>10}")
print("-" * 70)
for name, m in [
    ("Confidence (own threshold)", fair_conf_adv),
    ("Confidence (coverage-matched)", matched_conf_adv),
    ("Disagreement (own threshold)", fair_disagree_adv),
    ("Disagreement (coverage-matched)", matched_disagree_adv),
    ("Evidence (own threshold)", fair_evidence_adv),
    ("Evidence (coverage-matched)", matched_evidence_adv),
    ("Boundary distance (own threshold)", fair_boundary_adv),
]:
    print(f"{name:<45}{m['coverage']:>9.2f}%{m['accuracy']:>9.2f}%{m['risk']:>9.2f}%")

print("\nIf the coverage-matched rows still show meaningfully higher risk than boundary")
print("distance at the SAME coverage, that's real evidence boundary distance carries")
print("more information, not just a more conservative threshold. If they converge,")
print("most of boundary distance's apparent edge was coverage, not information.")

print("\nInterpretation:")
print(
    "At approximately matched adversarial coverage, "
    "multi-head disagreement and evidential evidence "
    "retain substantially lower risk than confidence "
    "and boundary distance."
)
print(
    "Boundary distance does not provide the strongest "
    "sensor under this experimental protocol."
)



Target adversarial coverage (boundary distance's own operating point): 71.50%
Coverage-matched thresholds (calibrated on calib_adv):
   Confidence:   0.2945  (vs. own threshold 0.3659)
   Disagreement: 0.000000284465  (vs. own threshold 0.000000281071)
   Evidence:     214.8941  (vs. own threshold 210.8431)

Method (adversarial, coverage-matched to ~72%)  Coverage  Accuracy      Risk
----------------------------------------------------------------------
Confidence (own threshold)                       75.75%    19.91%    80.09%
Confidence (coverage-matched)                    69.58%    19.28%    80.72%
Disagreement (own threshold)                     71.75%    78.98%    21.02%
Disagreement (coverage-matched)                  71.83%    78.89%    21.11%
Evidence (own threshold)                         72.75%    78.12%    21.88%
Evidence (coverage-matched)                      72.00%    78.36%    21.64%
Boundary distance (own threshold)                71.50%    21.68%    78.32%

If the c

In [ ]:
# -------------------------------------------------------------
# Part M: Publication figures
# These figures use only results already calculated above.
# -------------------------------------------------------------

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8-whitegrid")

FIGURE_DIR = "essay6_figures"
os.makedirs(FIGURE_DIR, exist_ok=True)


# -------------------------------------------------------------
# Figure 1: Same-sample risk comparison
# Primary figure for the essay.
# -------------------------------------------------------------

methods = [
    "Confidence",
    "Multi-head\nDisagreement",
    "Evidential\nEvidence",
    "Boundary\nDistance",
]

clean_risk = [
    fair_conf_clean["risk"],
    fair_disagree_clean["risk"],
    fair_evidence_clean["risk"],
    fair_boundary_clean["risk"],
]

adv_risk = [
    fair_conf_adv["risk"],
    fair_disagree_adv["risk"],
    fair_evidence_adv["risk"],
    fair_boundary_adv["risk"],
]

x = np.arange(len(methods))
width = 0.36

fig, ax = plt.subplots(figsize=(11, 6))

bars_clean = ax.bar(
    x - width / 2,
    clean_risk,
    width,
    label="Clean",
    color="#8FA8B8",
)

bars_adv = ax.bar(
    x + width / 2,
    adv_risk,
    width,
    label="Transferred PGD",
    color="#B85C5C",
)

ax.set_title(
    "Architectural signals retain lower risk under transferred attack",
    fontsize=15,
    pad=15,
)

ax.set_ylabel("Risk on retained predictions (%)")
ax.set_xticks(x)
ax.set_xticklabels(methods)
ax.set_ylim(0, 90)
ax.legend(frameon=False)

ax.bar_label(bars_clean, fmt="%.1f", padding=3, fontsize=9)
ax.bar_label(bars_adv, fmt="%.1f", padding=3, fontsize=9)

ax.text(
    0.02,
    -0.18,
    "Same 1,200 evaluation images with valid boundary measurements. "
    "Lower risk is better.",
    transform=ax.transAxes,
    fontsize=9,
    color="#555555",
)

fig.tight_layout()

fig.savefig(
    os.path.join(FIGURE_DIR, "fig1_same_sample_risk_comparison.png"),
    dpi=300,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)


# -------------------------------------------------------------
# Figure 2: Coverage versus adversarial risk
# This shows the operating trade-off, not only risk.
# -------------------------------------------------------------

adv_coverage = [
    fair_conf_adv["coverage"],
    fair_disagree_adv["coverage"],
    fair_evidence_adv["coverage"],
    fair_boundary_adv["coverage"],
]

fig, ax = plt.subplots(figsize=(9, 6))

colors = ["#B85C5C", "#4C7899", "#6A9A72", "#8A6A9A"]

for method, coverage, risk, color in zip(
    methods,
    adv_coverage,
    adv_risk,
    colors,
):
    ax.scatter(
        coverage,
        risk,
        s=180,
        color=color,
        edgecolor="black",
        linewidth=0.7,
        label=method.replace("\n", " "),
        zorder=3,
    )

    ax.annotate(
        method.replace("\n", " "),
        (coverage, risk),
        xytext=(7, 7),
        textcoords="offset points",
        fontsize=9,
    )

ax.set_title(
    "Adversarial risk at comparable coverage",
    fontsize=15,
    pad=15,
)

ax.set_xlabel("Adversarial coverage (%)")
ax.set_ylabel("Adversarial risk (%)")
ax.set_xlim(65, 80)
ax.set_ylim(0, 90)

ax.text(
    0.02,
    -0.16,
    "Same 1,200 evaluation images. Lower and farther left is more conservative; "
    "lower risk is the primary objective.",
    transform=ax.transAxes,
    fontsize=9,
    color="#555555",
)

fig.tight_layout()

fig.savefig(
    os.path.join(FIGURE_DIR, "fig2_coverage_risk_tradeoff.png"),
    dpi=300,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)


# -------------------------------------------------------------
# Figure 3: Marginal contribution of boundary distance
# This directly supports the fusion discussion.
# -------------------------------------------------------------

fusion_methods = [
    "Standalone\nconfidence",
    "Standalone\nmulti-head",
    "Standalone\nevidence",
    "Fused without\nboundary",
    "Fused with\nboundary",
]

fusion_adv_risk = [
    fair_conf_adv["risk"],
    fair_disagree_adv["risk"],
    fair_evidence_adv["risk"],
    fused_adv_nobd["risk"],
    fused_adv["risk"],
]

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(
    fusion_methods,
    fusion_adv_risk,
    color=[
        "#B85C5C",
        "#4C7899",
        "#6A9A72",
        "#9A9A9A",
        "#5F7771",
    ],
)

ax.set_title(
    "Boundary distance adds only a small marginal improvement to fusion",
    fontsize=15,
    pad=15,
)

ax.set_ylabel("Adversarial risk (%)")
ax.set_ylim(0, 90)

ax.bar_label(bars, fmt="%.1f", padding=3, fontsize=9)

ax.text(
    0.02,
    -0.18,
    "Fused rows use the full 8,000-image evaluation set; "
    "standalone rows use the shared 1,200-image comparison.",
    transform=ax.transAxes,
    fontsize=9,
    color="#555555",
)

fig.tight_layout()

fig.savefig(
    os.path.join(FIGURE_DIR, "fig3_boundary_marginal_contribution.png"),
    dpi=300,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

print(f"Saved publication figures to: {FIGURE_DIR}/")
